# Lab 02: Relational Algebra in Python

**Z2004 Database Management Systems (BS Core) · IIT Madras Zanzibar**

Executable implementations of select, project, natural join, and relational division.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z2004/labs/DB02_Relational_Algebra_Engine.ipynb)

Run cells top to bottom (`Runtime > Run all`). Built on Python `sqlite3` with full self-checks.

## 1. Relational Algebra Operators (`select`, `project`, `natural_join`, `division`)

Each operator maps relations (sets of tuples) to relations cleanly, making algebraic equivalences such as pushing selection before join transparent.

In [1]:
def select(rel, pred):
    return [dict(r) for r in rel if pred(r)]

def project(rel, cols):
    seen, out = set(), []
    for r in rel:
        t = tuple(r[c] for c in cols)
        if t not in seen:
            seen.add(t); out.append({c: r[c] for c in cols})
    return out

def natural_join(r1, r2):
    shared = [k for k in r1[0] if k in r2[0]]
    return [{**a, **b} for a in r1 for b in r2 if all(a[k] == b[k] for k in shared)]

def division(r_xy, s_y, x_cols, y_cols):
    req = {tuple(s[c] for c in y_cols) for s in s_y}
    groups = {}
    for row in r_xy:
        groups.setdefault(tuple(row[c] for c in x_cols), set()).add(tuple(row[c] for c in y_cols))
    return [dict(zip(x_cols, xk)) for xk, ys in groups.items() if req.issubset(ys)]

students = [{"SID": 1, "Name": "Asha", "Dept": "CSE"}, {"SID": 2, "Name": "Omar", "Dept": "CSE"}, {"SID": 3, "Name": "Zahra", "Dept": "EEE"}]
enrol = [{"SID": 1, "CID": "DB101"}, {"SID": 1, "CID": "CS102"}, {"SID": 2, "CID": "DB101"}, {"SID": 3, "CID": "EE105"}]
core = [{"CID": "DB101"}, {"CID": "CS102"}]

print("CSE students:", project(select(students, lambda r: r["Dept"] == "CSE"), ["Name"]))
all_core = natural_join(students, division(enrol, core, ["SID"], ["CID"]))
print("Students enrolled in ALL core courses:", all_core)
assert [r["Name"] for r in all_core] == ["Asha"]


CSE students: [{'Name': 'Asha'}, {'Name': 'Omar'}]
Students enrolled in ALL core courses: [{'SID': 1, 'Name': 'Asha', 'Dept': 'CSE'}]
